In [1]:
%pip install -q -r ../requirements.txt
from pathlib import Path
from dotenv import load_dotenv
import sys

if "../src" not in sys.path:
    sys.path.append("../src")

load_dotenv(Path("../.env").resolve())

Note: you may need to restart the kernel to use updated packages.


False

In [2]:
from domain import Base, RecipeRepository, Session, UserRepository, engine #type: ignore
from domain.models import Recipe, User #type: ignore

engine.url

RuntimeError: Database URL is not configured. Set DATABASE_URL or DB_URL_DEVELOPMENT/DB_URL_TEST/DB_URL_PRODUCTION.

In [3]:
Base.metadata.create_all(engine)

print(Base.metadata.tables.keys())

dict_keys(['category', 'favorite', 'follow', 'ingredient', 'rating', 'recipe', 'recipe_category', 'recipe_image', 'recipe_ingredient', 'recipe_step', 'users'])


In [4]:
session = Session()

user = User(
    username="Ayoube E",
    email="Ayoube3@example.com",
    password_hash="Secret Pass1",
)
user_repository = UserRepository(session)
user_repository.save(user)

session.flush()

recipe = Recipe(
    author=user,
    title="Salmon con patatas",
    description="Simple Recipe",
)
recipe_repository = RecipeRepository(session)
recipe_repository.save(recipe)

session.commit()
print(user.user_id, recipe.recipe_id)

20 11


In [5]:
from sqlalchemy import select

stmt = select(Recipe)
recipes = session.scalars(stmt).all()

for recipe in recipes:
    print(f"{recipe.recipe_id} - {recipe.description}")

1 - Simple pasta recipe
2 - Simple pasta recipe2
3 - Simple pasta recipe2
4 - Simple pasta recipe2
5 - Simple pasta recipe2
6 - Simple pasta recipe2
7 - Simple pasta recipe2
8 - Simple pasta recipe2
9 - Simple pasta recipe2
10 - Simple pasta recipe2
11 - Simple Recipe


In [6]:
from sqlalchemy import select

stmt = select(User)
users = session.scalars(stmt).all()

for user in users:
    print(user)
    for recipe in user.recipes:
         print(f"   {recipe.recipe_id} - {recipe.description}")

User(user_id=1, username='chef_anna', email='anna@example.com')
   1 - Simple pasta recipe
User(user_id=3, username='chef1_anna', email='anna1@example.com')
   2 - Simple pasta recipe2
User(user_id=5, username='chef1_anna', email='anna2@example.com')
   3 - Simple pasta recipe2
User(user_id=10, username='chef2_anna', email='chef2_anna@example.com')
   5 - Simple pasta recipe2
User(user_id=11, username='chef3_anna', email='chef3_anna@example.com')
   6 - Simple pasta recipe2
User(user_id=13, username='chef4_anna', email='chef4_anna@example.com')
   7 - Simple pasta recipe2
User(user_id=15, username='chef4_anna', email='Ayoube@example.com')
   8 - Simple pasta recipe2
User(user_id=17, username='chef4_anna', email='Ayoube1@example.com')
   9 - Simple pasta recipe2
User(user_id=19, username='chef5_anna', email='Ayoube2@example.com')
   10 - Simple pasta recipe2
User(user_id=8, username='chef1_anna', email='anna6@uvic.cat')
   4 - Simple pasta recipe2
User(user_id=20, username='Ayoube E', e

In [7]:
class UnitOfWork:
    def __init__(self, session_factory):
        self.session_factory = session_factory
        self.session = None

    def __enter__(self):
        self.session = self.session_factory()
        return self.session

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            self.session.commit()
        else:
            self.session.rollback()
        self.session.close()

with UnitOfWork(Session) as uow_session:
    print(uow_session)